In [1]:
import geopandas as gp
import pandas as pd
import os
import numpy as np
import re
from collections import Counter
import pber_functions_v1 as pber

# Alabama 2024 Primary Election Results

In [2]:
results = pd.read_csv("./al_2024_prim_prec_csv/al_2024_prim_prec_csv.csv")

In [3]:
jefferson = results[results["County"]=="Jefferson"].copy()
others = results[results["County"]!="Jefferson"].copy()
races = [i for i in list(results.columns) if i not in ["UNIQUE_ID","COUNTYFP","County","Precinct"]]

In [4]:
jefferson_absentee = jefferson[jefferson["Precinct"].str.contains("ABSENTEE") | jefferson["Precinct"].str.contains("PROVISIONAL")].copy(deep = True)

In [5]:
jefferson_absentee

,UNIQUE_ID,COUNTYFP,County,Precinct,P24A01NO,P24A01YES,P24CFJRSTE,P24CFJRTAY,P24CR2RAND,P24CR2RGOV,...,PCON06RPAL,PCON06RWIL,PCON07DDAV,PCON07DSEW,PCON07RHOR,PCON07RLIT,RCON02DDAN,RCON02DFIG,RCON02RBRE,RCON02RDOB
960,073-:-BESSEMER ABSENTEE,73,Jefferson,BESSEMER ABSENTEE,321,173,42,32,25,29,...,5,1,16,428,20,32,0,0,0,0
961,073-:-BIRMINGHAM ABSENTEE,73,Jefferson,BIRMINGHAM ABSENTEE,451,402,173,97,118,120,...,192,27,107,448,25,15,0,0,0,0
1137,073-:-PROVISIONAL,73,Jefferson,PROVISIONAL,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
jefferson_absentee_divs = jefferson_absentee[jefferson_absentee["Precinct"].str.contains("BESSEMER") | jefferson_absentee["Precinct"].str.contains("BIRMINGHAM")].copy(deep = True)
jefferson_absentee_cnty = jefferson_absentee[~jefferson_absentee["Precinct"].str.contains("BESSEMER") & ~jefferson_absentee["Precinct"].str.contains("BIRMINGHAM")].copy(deep = True)


In [7]:
jefferson_absentee_divs_bess = jefferson_absentee_divs[jefferson_absentee_divs["Precinct"].str.contains("BESSEMER")].copy(deep = True)
jefferson_absentee_divs_birm = jefferson_absentee_divs[jefferson_absentee_divs["Precinct"].str.contains("BIRMINGHAM")].copy(deep = True)

In [8]:
jefferson_allocate = jefferson[~(jefferson["Precinct"].str.contains("ABSENTEE")) & ~(jefferson["Precinct"].str.contains("PROVISIONAL"))].copy(deep = True)

In [9]:
jefferson_allocate

,UNIQUE_ID,COUNTYFP,County,Precinct,P24A01NO,P24A01YES,P24CFJRSTE,P24CFJRTAY,P24CR2RAND,P24CR2RGOV,...,PCON06RPAL,PCON06RWIL,PCON07DDAV,PCON07DSEW,PCON07RHOR,PCON07RLIT,RCON02DDAN,RCON02DFIG,RCON02RBRE,RCON02RDOB
962,073-:-PREC 1010 - HUFFMAN BAPTIST CH,73,Jefferson,PREC 1010 - HUFFMAN BAPTIST CH,578,565,174,104,114,135,...,0,0,51,861,148,99,0,0,0,0
963,073-:-PREC 1020 - TOM BRADFORD PARK,73,Jefferson,PREC 1020 - TOM BRADFORD PARK,446,383,160,67,84,117,...,214,13,0,0,0,0,0,0,0,0
964,073-:-PREC 1030 - L_M_ SMITH MIDDLE,73,Jefferson,PREC 1030 - L_M_ SMITH MIDDLE,450,213,51,12,28,32,...,0,0,22,588,35,23,0,0,0,0
965,073-:-PREC 1040 - BETHEL BAPTIST CHU,73,Jefferson,PREC 1040 - BETHEL BAPTIST CHU,595,243,8,4,4,7,...,0,0,48,799,8,3,0,0,0,0
966,073-:-PREC 1050 - MIDFIELD COMMUNITY,73,Jefferson,PREC 1050 - MIDFIELD COMMUNITY,384,177,36,10,19,24,...,0,0,23,493,29,13,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1132,073-:-PREC 5230 - HOOVER PUBLIC LIBR,73,Jefferson,PREC 5230 - HOOVER PUBLIC LIBR,137,172,127,100,88,95,...,196,23,0,0,0,0,0,0,0,0
1133,073-:-PREC 5240 - SHADES CAHABA ELEM,73,Jefferson,PREC 5240 - SHADES CAHABA ELEM,173,273,177,142,108,147,...,257,46,0,0,0,0,0,0,0,0
1134,073-:-PREC 5250 - CHURCH OF THE HIGH,73,Jefferson,PREC 5250 - CHURCH OF THE HIGH,210,289,201,188,130,194,...,322,57,0,0,0,0,0,0,0,0
1135,073-:-PREC 5260 - CANTERBURY UNITED,73,Jefferson,PREC 5260 - CANTERBURY UNITED,241,418,321,273,158,330,...,509,62,0,0,0,0,0,0,0,0


In [10]:
#jefferson_allocate[["UNIQUE_ID","County"]].to_csv("./jefferson_precs.csv", index = False)

In [11]:
jefferson_divisions = pd.read_csv("./raw-from-source/jefferson_divisions.csv")
jefferson_divisions.drop(["County"], axis = 1, inplace = True)

In [12]:
jefferson_divisions["Division"].value_counts()

Division
BIRMINGHAM    121
BESSEMER       43
BOTH           11
Name: count, dtype: int64

In [13]:
jefferson_divisions["Division2"] = np.where(jefferson_divisions["Division"]=="BOTH","BIRMINGHAM",jefferson_divisions["Division"])
jefferson_divisions["Division1"] = np.where(jefferson_divisions["Division"]=="BOTH","BESSEMER",jefferson_divisions["Division"])

In [14]:
jefferson_allocate_full = pd.merge(jefferson_allocate,jefferson_divisions,on="UNIQUE_ID", how = "outer", indicator = True)

In [15]:
jefferson_allocate_full.drop("_merge", axis = 1, inplace = True)

In [16]:
birmingham_precs = jefferson_allocate_full[jefferson_allocate_full["Division2"]=="BIRMINGHAM"].copy(deep = True)
bessemer_precs = jefferson_allocate_full[jefferson_allocate_full["Division1"]=="BESSEMER"].copy(deep = True)

In [17]:
birmingham_precs_allocated = pber.allocate_absentee(birmingham_precs, jefferson_absentee_divs_birm, races, "COUNTYFP")
bessemer_precs_allocated = pber.allocate_absentee(bessemer_precs, jefferson_absentee_divs_bess, races, "COUNTYFP")

/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:234: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_allocate = int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==race_district][race])
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:251: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_receiving_votes.loc[:,rem_var]=0.0
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:252: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once u

/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0

/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:251: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_receiving_votes.loc[:,rem_var]=0.0
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:252: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_receiving_votes.loc[:,floor_var]=0.0
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:250: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` 

/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0

In [18]:
for prec in set(birmingham_precs_allocated[birmingham_precs_allocated["Division"]=="BOTH"]["UNIQUE_ID"]):
    for race in races:
        print(birmingham_precs_allocated.loc[birmingham_precs_allocated["UNIQUE_ID"]==prec,race].values[0])
        print(results.loc[results["UNIQUE_ID"]==prec,race].values[0])
        birmingham_precs_allocated.loc[birmingham_precs_allocated["UNIQUE_ID"]==prec,race] -= results.loc[results["UNIQUE_ID"]==prec,race].values[0]
        print(birmingham_precs_allocated.loc[birmingham_precs_allocated["UNIQUE_ID"]==prec,race].values[0])
        print("")

326
322
4

245
242
3

76
75
1

60
60
0

61
60
1

52
52
0

69
68
1

49
49
0

427
418
9

17
16
1

15
15
0

0
0
0

0
0
0

1
1
0

59
58
1

2
2
0

0
0
0

94
93
1

2
2
0

64
63
1

67
67
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

39
36
3

421
412
9

65
64
1

48
48
0

0
0
0

0
0
0

0
0
0

0
0
0

130
128
2

55
54
1

0
0
0

1
1
0

0
0
0

1
1
0

1
1
0

0
0
0

182
178
4

5
5
0

4
4
0

0
0
0

0
0
0

0
0
0

1
1
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

1
1
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

7
7
0

183
179
4

1
1
0

0
0
0

0
0
0

0
0
0

0
0
0

0
0
0

290
286
4

362
357
5

326
323
3

247
245
2

228
226
2

246
244
2

277
274
3

190
189
1

79
77
2

3
3
0

14
14
0

1
1
0

1
1
0

1

In [19]:
jefferson_final = pd.concat([bessemer_precs_allocated,birmingham_precs_allocated])

In [20]:
jefferson_final.drop(["Division","Division2","Division1"], axis = 1, inplace = True)

In [21]:
for race in races:
    print(race)
    jefferson_final[race] = jefferson_final[race].astype(int)

P24A01NO
P24A01YES
P24CFJRSTE
P24CFJRTAY
P24CR2RAND
P24CR2RGOV
P24CV2RHAN
P24CV2RPAR
P24PREDBID
P24PREDPHI
P24PREDUNC
P24PRERBIN
P24PRERCHR
P24PRERDES
P24PRERHAL
P24PRERRAM
P24PRERSTU
P24PRERTRU
P24PRERUNC
P24PSCRCAV
P24PSCRMCC
PCON01RCAR
PCON01RMOO
PCON02DAVE
PCON02DBRA
PCON02DCOL
PCON02DDAN
PCON02DFIG
PCON02DGIV
PCON02DGRA
PCON02DHAR
PCON02DLEN
PCON02DPAT
PCON02DSIM
PCON02RALB
PCON02RBRE
PCON02RDOB
PCON02RDUP
PCON02RGIL
PCON02RHAR
PCON02RSHE
PCON02RTHO
PCON03RBEV
PCON03RNEW
PCON03RROG
PCON04RADE
PCON04RHOL
PCON06RMCF
PCON06RPAL
PCON06RWIL
PCON07DDAV
PCON07DSEW
PCON07RHOR
PCON07RLIT
RCON02DDAN
RCON02DFIG
RCON02RBRE
RCON02RDOB


In [22]:
jefferson_final = jefferson_final.groupby(["UNIQUE_ID","COUNTYFP","County","Precinct"], as_index = False).sum()

In [23]:
others = pd.concat([others, jefferson_final, jefferson_absentee_cnty])

In [24]:
others.reset_index(inplace = True, drop = True)

In [25]:
others_allocate = others[~(others["Precinct"].str.contains("ABSENTEE")) & ~(others["Precinct"].str.contains("PROVISIONAL")) & ~(others["Precinct"].str.contains("PROVSIONAL")) & ~(others["Precinct"].str.contains("PROVISONAL"))].copy(deep = True)

In [26]:
others_absentee = others[(others["Precinct"].str.contains("ABSENTEE")) | (others["Precinct"].str.contains("PROVISIONAL")) | (others["Precinct"].str.contains("PROVSIONAL")) | (others["Precinct"].str.contains("PROVISONAL"))].copy(deep = True)

In [27]:
others_allocated = pber.allocate_absentee(others_allocate, others_absentee, races, "COUNTYFP")

Special allocation used for

/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:234: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_allocate = int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==race_district][race])
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:251: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_receiving_votes.loc[:,rem_var]=0.0
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:252: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once u

 [[91, 'P24PRERBIN']]


/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0

/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0

...

/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0

/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0

/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0

/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0

/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  to_go = int(np.round((int(to_dole_out_totals.loc[to_dole_out_totals[col_allocating]==county][race])-first_allocation.loc[first_allocation.index==county,floor_var])))
/Users/peterhorton/Documents/RDH/pber_local/AL_2024/primary/pber_functions_v1.py:303: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0

In [28]:
others_allocated

,UNIQUE_ID,COUNTYFP,County,Precinct,P24A01NO,P24A01YES,P24CFJRSTE,P24CFJRTAY,P24CR2RAND,P24CR2RGOV,...,PCON06RPAL,PCON06RWIL,PCON07DDAV,PCON07DSEW,PCON07RHOR,PCON07RLIT,RCON02DDAN,RCON02DFIG,RCON02RBRE,RCON02RDOB
0,001-:-10 JONES COMM_ CTR_,1,Autauga,10 JONES COMM_ CTR_,70,73,63,55,64,45,...,86,20,0,0,0,0,0,0,0,0
1,001-:-100 TRINITY METHODIST,1,Autauga,100 TRINITY METHODIST,445,391,449,386,364,308,...,616,123,0,0,0,0,0,0,0,0
2,001-:-110 CENTRAL AL ELECTRIC,1,Autauga,110 CENTRAL AL ELECTRIC,47,71,40,20,24,27,...,52,4,0,0,0,0,0,0,0,0
3,001-:-140 AUTAUGAVILLE VFD,1,Autauga,140 AUTAUGAVILLE VFD,135,134,125,78,87,89,...,167,19,0,0,0,0,0,0,0,0
4,001-:-150 PRATTMONT BAPTIST,1,Autauga,150 PRATTMONT BAPTIST,143,159,155,116,136,103,...,183,65,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2105,073-:-PREC 5230 - HOOVER PUBLIC LIBR,73,Jefferson,PREC 5230 - HOOVER PUBLIC LIBR,139,174,128,101,89,96,...,198,23,0,0,0,0,0,0,0,0
2106,073-:-PREC 5240 - SHADES CAHABA ELEM,73,Jefferson,PREC 5240 - SHADES CAHABA ELEM,175,277,179,143,109,148,...,259,47,0,0,0,0,0,0,0,0
2107,073-:-PREC 5250 - CHURCH OF THE HIGH,73,Jefferson,PREC 5250 - CHURCH OF THE HIGH,213,293,203,189,131,195,...,325,58,0,0,0,0,0,0,0,0
2108,073-:-PREC 5260 - CANTERBURY UNITED,73,Jefferson,PREC 5260 - CANTERBURY UNITED,244,424,324,275,159,333,...,513,63,0,0,0,0,0,0,0,0


In [29]:
for race in races:
    others_allocated[race] = others_allocated[race].astype(int)
    results[race] = results[race].astype(int)
    print(sum(others_allocated[race])-sum(results[race]))

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [30]:
pber.county_totals_check(results, "Pre-Alloc", others_allocated, "Post-Alloc", races, "County", full_print=False, method='county')

***Countywide Totals Check***

Counties that match:

['Autauga', 'Baldwin', 'Barbour', 'Bibb', 'Blount', 'Bullock', 'Butler', 'Calhoun', 'Chambers', 'Cherokee', 'Chilton', 'Choctaw', 'Clarke', 'Clay', 'Cleburne', 'Coffee', 'Colbert', 'Conecuh', 'Coosa', 'Covington', 'Crenshaw', 'Cullman', 'Dale', 'Dallas', 'DeKalb', 'Elmore', 'Escambia', 'Etowah', 'Fayette', 'Franklin', 'Geneva', 'Greene', 'Hale', 'Henry', 'Houston', 'Jackson', 'Jefferson', 'Lamar', 'Lauderdale', 'Lawrence', 'Lee', 'Limestone', 'Lowndes', 'Macon', 'Madison', 'Marengo', 'Marion', 'Marshall', 'Mobile', 'Monroe', 'Montgomery', 'Morgan', 'Perry', 'Pickens', 'Pike', 'Randolph', 'Russell', 'Shelby', 'St. Clair', 'Sumter', 'Talladega', 'Tallapoosa', 'Tuscaloosa', 'Walker', 'Washington', 'Wilcox', 'Winston']


In [31]:
others_allocated["COUNTYFP"] = others_allocated["COUNTYFP"].astype(str).str.zfill(3)

In [32]:
others_allocated["UNIQUE_ID"] = others_allocated["County"] + "-:-" + others_allocated["Precinct"]

In [33]:
others_allocated.to_csv("./al_2024_prim_prec_allocated.csv", index = False)

In [34]:
others_allocated

,UNIQUE_ID,COUNTYFP,County,Precinct,P24A01NO,P24A01YES,P24CFJRSTE,P24CFJRTAY,P24CR2RAND,P24CR2RGOV,...,PCON06RPAL,PCON06RWIL,PCON07DDAV,PCON07DSEW,PCON07RHOR,PCON07RLIT,RCON02DDAN,RCON02DFIG,RCON02RBRE,RCON02RDOB
0,Autauga-:-10 JONES COMM_ CTR_,001,Autauga,10 JONES COMM_ CTR_,70,73,63,55,64,45,...,86,20,0,0,0,0,0,0,0,0
1,Autauga-:-100 TRINITY METHODIST,001,Autauga,100 TRINITY METHODIST,445,391,449,386,364,308,...,616,123,0,0,0,0,0,0,0,0
2,Autauga-:-110 CENTRAL AL ELECTRIC,001,Autauga,110 CENTRAL AL ELECTRIC,47,71,40,20,24,27,...,52,4,0,0,0,0,0,0,0,0
3,Autauga-:-140 AUTAUGAVILLE VFD,001,Autauga,140 AUTAUGAVILLE VFD,135,134,125,78,87,89,...,167,19,0,0,0,0,0,0,0,0
4,Autauga-:-150 PRATTMONT BAPTIST,001,Autauga,150 PRATTMONT BAPTIST,143,159,155,116,136,103,...,183,65,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2105,Jefferson-:-PREC 5230 - HOOVER PUBLIC LIBR,073,Jefferson,PREC 5230 - HOOVER PUBLIC LIBR,139,174,128,101,89,96,...,198,23,0,0,0,0,0,0,0,0
2106,Jefferson-:-PREC 5240 - SHADES CAHABA ELEM,073,Jefferson,PREC 5240 - SHADES CAHABA ELEM,175,277,179,143,109,148,...,259,47,0,0,0,0,0,0,0,0
2107,Jefferson-:-PREC 5250 - CHURCH OF THE HIGH,073,Jefferson,PREC 5250 - CHURCH OF THE HIGH,213,293,203,189,131,195,...,325,58,0,0,0,0,0,0,0,0
2108,Jefferson-:-PREC 5260 - CANTERBURY UNITED,073,Jefferson,PREC 5260 - CANTERBURY UNITED,244,424,324,275,159,333,...,513,63,0,0,0,0,0,0,0,0


In [35]:
set(others_allocated["Precinct"])

{'CHURCH AT CAHABA BEND',
 'PREC 2320 - UNITARIAN UNIVERSA',
 'ANDERSON GYMNASIUM',
 'MOUNTAINBORO VFD',
 'BELGREEN FIRE DEPT_',
 'COUNTY LINE VFD CONG 3',
 'CROSSVILLE SAND MTN RSCH',
 'PREC 5005 - BIRMINGHAM FIRST S',
 'LIVINGSTON CIVIC CTR',
 'LIBERTY BAPTIST CH',
 'MALVERN COMM_ CTR_',
 'REDHILL COMM_ CTR_',
 'HIGHTOGY RIDGE',
 'GLENWOOD SCHOOL GYM',
 'CREOLA SENIOR CTR',
 'MAGNOLIA COMM_ CTR_',
 'PINE RIDGE COMM D-4',
 'GREENPOND FIRE DEPT',
 "PREC 1380 - ST_ MARY'S CATHOLI",
 'CAVE SPRINGS COMM_',
 'MCCARLEY CTR_ 6-1',
 'ELKMONT FIRE DEPT_',
 'UNION FREEWILL BAPTIST',
 'ROCK MILLS',
 'THE WHITESBURG CTR_',
 'PINE APPLE LIBRARY',
 'MOVEMENT CH',
 'SUMMIT',
 'DOTHAN 1ST ASSEMBLY CLC',
 'WHITE PLAINS VFD',
 'STRAIGHT MOUNTAIN',
 'CHURNTOWN',
 'GEIGER TOWN HALL',
 'YELLOW BLUFF CITY HALL',
 'BLAKE COMM CTR',
 'TABERNACLE OF PRAISE',
 'REGENCY CH OF CHRIST',
 'PREC 4025 - TRUSSVILLE CIVIC C',
 'SPRING CREEK VFD',
 'NEW FRANKLIN FIRE DEPT',
 'STRAUGHN HIGH SCHOOL GYM',
 'SHORTERVILLE B

In [36]:
results = pd.read_csv("./al_2024_prim_prec_csv/al_2024_prim_prec_csv.csv")

In [37]:
others_allocated = pd.read_csv("./raw-from-source/al_2024_prim_prec_allocated.csv")

In [38]:
others_allocated.sum()

UNIQUE_ID     Autauga-:-10 JONES COMM_ CTR_Autauga-:-100 TRI...
COUNTYFP                                                 139394
County        AutaugaAutaugaAutaugaAutaugaAutaugaAutaugaAuta...
Precinct      10 JONES COMM_ CTR_100 TRINITY METHODIST110 CE...
P24A01NO                                                 360944
                                    ...                        
PCON07RLIT                                                12986
RCON02DDAN                                                14006
RCON02DFIG                                                21962
RCON02RBRE                                                10471
RCON02RDOB                                                14705
Length: 62, dtype: object

In [39]:
for race in races:
    print(results.sum()[race]-others_allocated.sum()[race])

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [40]:
others_allocated.sum()

UNIQUE_ID     Autauga-:-10 JONES COMM_ CTR_Autauga-:-100 TRI...
COUNTYFP                                                 139394
County        AutaugaAutaugaAutaugaAutaugaAutaugaAutaugaAuta...
Precinct      10 JONES COMM_ CTR_100 TRINITY METHODIST110 CE...
P24A01NO                                                 360944
                                    ...                        
PCON07RLIT                                                12986
RCON02DDAN                                                14006
RCON02DFIG                                                21962
RCON02RBRE                                                10471
RCON02RDOB                                                14705
Length: 62, dtype: object